In [2]:
import pandas as pd
EXCEPTION_NUMBER=['451','458','480']
meta=pd.read_csv(r"D:\depression_dataset(DAIC-WOZ)\metadataset.csv")

meta

#nr.reduce_noise(y=y_raw, sr=sr, stationary=True, prop_decrease=0.8)
#세그 먼트에서 반드시 이걸 거친 뒤에, 라벨링을 하고 기본적으로 모델은 BILSTM 으로 하되
# 음성 추출은 일단 MFCC를 추출해서 LSTM에 집어 넣을 것
# 시계열의 방향은 각 세그 먼트 
# 세그먼트1 -> 세그먼트2 이중 BILSTM을 한다면 과연 효과가 있을 것 인가? 한 세그먼트의 흘러감의 정보를 파악하고
# 그다음 전체 세그먼트간의 그 흐름 이동에서 의미가 있을 것인지? , 세그먼트 별로 발화길이가 다른데 패딩 말고는
# 방법론이 크게 없는지 ? 그리고, 질문 종류 까지 넣긴 할 것, 나랑 함께 의논을 하고 코드를 만드는걸로
# 일단 음성 mfcc 적용

,Participant_ID,Binary,Score,Gender,Transcript,Audio,Group
0,300,0,2,1,D:\depression_dataset(DAIC-WOZ)\300_P\300_clea...,D:\depression_dataset(DAIC-WOZ)\300_P\300_AUDI...,Test
1,301,0,3,1,D:\depression_dataset(DAIC-WOZ)\301_P\301_clea...,D:\depression_dataset(DAIC-WOZ)\301_P\301_AUDI...,Test
2,302,0,4,1,D:\depression_dataset(DAIC-WOZ)\302_P\302_clea...,D:\depression_dataset(DAIC-WOZ)\302_P\302_AUDI...,Validation
3,303,0,0,0,D:\depression_dataset(DAIC-WOZ)\303_P\303_clea...,D:\depression_dataset(DAIC-WOZ)\303_P\303_AUDI...,Train
4,304,0,6,0,D:\depression_dataset(DAIC-WOZ)\304_P\304_clea...,D:\depression_dataset(DAIC-WOZ)\304_P\304_AUDI...,Train
...,...,...,...,...,...,...,...
181,488,0,0,0,D:\depression_dataset(DAIC-WOZ)\488_P\488_clea...,D:\depression_dataset(DAIC-WOZ)\488_P\488_AUDI...,Train
182,489,0,3,1,D:\depression_dataset(DAIC-WOZ)\489_P\489_clea...,D:\depression_dataset(DAIC-WOZ)\489_P\489_AUDI...,Validation
183,490,0,2,1,D:\depression_dataset(DAIC-WOZ)\490_P\490_clea...,D:\depression_dataset(DAIC-WOZ)\490_P\490_AUDI...,Validation
184,491,0,8,0,D:\depression_dataset(DAIC-WOZ)\491_P\491_clea...,D:\depression_dataset(DAIC-WOZ)\491_P\491_AUDI...,Train


**데이터 전처리, 방법론, 제외할 것, 데이터 품질 측정 더 확실하게?**

In [20]:
import pandas as pd
import numpy as np
import os
import torch
import librosa
import noisereduce as nr
from tqdm import tqdm
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import warnings
import pickle
from collections import defaultdict

warnings.filterwarnings('ignore')

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
EXCEPTION_NUMBER = [
    '451', '458', '480'           # Ellie 발화 누락 (3명)
]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 하이퍼파라미터
MAX_UTTERANCE_DURATION = 15.0   # 최대 발화 길이 (초) - 이상 분할
MIN_UTTERANCE_DURATION = 0.5    # 최소 발화 길이 (초) - 이하 제거
SR = 16000

# Question Type 매핑 (단순화)
Q_TYPE_MAPPING = {
    'casual': 0,      # small talk, preference, open-ended encouragement
    'background': 1,  # daily habits, social, self-perception
    'emotional': 2,   # emotion / mood
    'clinical': 3,    # depression symptoms direct
    'other': 4
}

# 원본 → 단순화 매핑
Q_TYPE_SIMPLIFICATION = {
    'small talk': 'casual',
    'preference': 'casual',
    'open-ended encouragement': 'casual',
    
    'daily habits / lifestyle': 'background',
    'social / family / relationship': 'background',
    'self-perception / personality': 'background',
    
    'emotion / mood': 'emotional',
    
    'depression symptoms direct': 'clinical',
    
    'other': 'other'
}

print(f"⏳ Wav2Vec 2.0 모델 로딩 중... (Device: {DEVICE})")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)
model.eval()
print("✅ 모델 로드 완료!")


# =============================================================================
# 텍스트 특징 (TTR) - 발화 단위
# =============================================================================
def get_ttr(text):
    """Type-Token Ratio 계산"""
    if not text or len(text.strip()) == 0:
        return 0.0
    tokens = text.lower().split()
    if len(tokens) == 0:
        return 0.0
    return len(set(tokens)) / len(tokens)


# =============================================================================
# Wav2Vec 특징 추출
# =============================================================================
def extract_wav2vec(y, sr):
    """Wav2Vec 2.0 특징 추출"""
    try:
        if len(y) < SR * 0.3:  # 0.3초 미만은 너무 짧음
            return None
        
        inputs = processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(DEVICE)
        
        with torch.no_grad():
            outputs = model(input_values)
            hidden_states = outputs.last_hidden_state  # [1, time_steps, 768]
        
        # 평균 풀링
        embedding = torch.mean(hidden_states, dim=1).squeeze().cpu().numpy()
        
        # 안전성 체크
        if not np.isfinite(embedding).all():
            return None
        
        return embedding
    except Exception as e:
        return None


# =============================================================================
# 대화 전처리 클래스
# =============================================================================
class UtterancePreprocessor:
    def __init__(self, base_path):
        self.base_path = base_path
    
    def normalize_question_type(self, q_type):
        """질문 유형 정규화 및 단순화"""
        q_type = q_type.lower().strip()
        q_type = ' '.join(q_type.split())  # 공백 정규화
        q_type = q_type.replace('/', ' / ')
        q_type = ' '.join(q_type.split())
        
        # 단순화
        if q_type in Q_TYPE_SIMPLIFICATION:
            return Q_TYPE_SIMPLIFICATION[q_type]
        return 'other'
    
    def process_transcript(self, pid):
        """CSV에서 발화 단위로 추출"""
        transcript_path = os.path.join(self.base_path, f"{pid}_P", f"{pid}_cleaned_transcript.csv")
        
        try:
            df = pd.read_csv(transcript_path, sep='\t')
            if df.shape[1] < 2:
                df = pd.read_csv(transcript_path, sep=',')
        except:
            return []
        
        # 컬럼명 정규화
        df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
        
        # question_label 처리
        if 'question_label' not in df.columns:
            df['question_label'] = 'other'
        df['question_label'] = df['question_label'].fillna('other')
        df['question_label'] = df['question_label'].replace('', 'other')
        
        # 발화 추출
        utterances = self._extract_utterances(df, pid)
        
        # 긴 발화 분할
        final_utterances = self._split_long_utterances(utterances)
        
        return final_utterances
    
    def _extract_utterances(self, df, pid):
        """발화 단위로 추출 (합치지 않음)"""
        utterances = []
        current_q_type = None
        first_ellie_found = False
        
        for idx, row in df.iterrows():
            speaker = str(row['speaker']).strip().lower()
            q_label = str(row['question_label']).strip().lower()
            
            # Ellie 발화
            if 'ellie' in speaker:
                first_ellie_found = True
                
                # 질문 유형 업데이트
                q_label = self.normalize_question_type(q_label)
                if q_label != 'other':
                    current_q_type = q_label
                else:
                    current_q_type = 'other'
            
            # Participant 발화
            elif 'participant' in speaker:
                if not first_ellie_found:
                    continue  # 첫 Ellie 발화 전 무시
                
                if current_q_type is None:
                    continue
                
                text = str(row['value'])
                start_time = row['start_time']
                stop_time = row['stop_time']
                duration = stop_time - start_time
                
                # 너무 짧은 발화 제거
                if duration < MIN_UTTERANCE_DURATION:
                    continue
                
                utterances.append({
                    'pid': pid,
                    'q_type': current_q_type,
                    'text': text,
                    'start': start_time,
                    'end': stop_time,
                    'duration': duration
                })
        
        return utterances
    
    def _split_long_utterances(self, utterances):
        """긴 발화를 15초 단위로 분할"""
        final_utterances = []
        
        for utt in utterances:
            duration = utt['duration']
            
            if duration <= MAX_UTTERANCE_DURATION:
                final_utterances.append(utt)
            else:
                # 분할
                num_splits = int(np.ceil(duration / MAX_UTTERANCE_DURATION))
                split_duration = duration / num_splits
                
                for i in range(num_splits):
                    split_start = utt['start'] + i * split_duration
                    split_end = min(split_start + split_duration, utt['end'])
                    
                    final_utterances.append({
                        'pid': utt['pid'],
                        'q_type': utt['q_type'],
                        'text': utt['text'],  # 텍스트는 동일하게 유지
                        'start': split_start,
                        'end': split_end,
                        'duration': split_end - split_start
                    })
        
        return final_utterances


# =============================================================================
# 오디오 품질 검증
# =============================================================================
def check_audio_quality(y, sr):
    """오디오 품질 검사"""
    issues = []
    
    # 1. 무음 비율 체크 (80% 이상 무음이면 문제)
    energy = librosa.feature.rms(y=y)[0]
    silence_ratio = np.sum(energy < 0.01) / len(energy)
    if silence_ratio > 0.8:
        issues.append(f"무음 비율 높음: {silence_ratio:.2%}")
    
    # 2. 클리핑 체크 (진폭이 0.99 이상인 비율)
    clipping_ratio = np.sum(np.abs(y) > 0.99) / len(y)
    if clipping_ratio > 0.01:
        issues.append(f"클리핑 발생: {clipping_ratio:.2%}")
    
    # 3. 너무 작은 볼륨
    max_amplitude = np.max(np.abs(y))
    if max_amplitude < 0.01:
        issues.append(f"볼륨 너무 작음: {max_amplitude:.4f}")
    
    return issues


# =============================================================================
# 메인 파이프라인
# =============================================================================
def run_preprocessing_pipeline():
    """전체 전처리 파이프라인 실행"""
    # 메타데이터 로드
    meta = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta['Participant_ID'] = meta['Participant_ID'].astype(str)
    meta = meta[~meta['Participant_ID'].isin(EXCEPTION_NUMBER)].reset_index(drop=True)
    
    preprocessor = UtterancePreprocessor(BASE_PATH)
    
    # 참가자별 데이터 저장
    dataset = {}
    
    print(f"\n{'='*70}")
    print(f"🚀 전처리 시작: {len(meta)}명")
    print(f"{'='*70}\n")
    
    stats = {
        'processed': 0,
        'failed': 0,
        'total_utterances': 0,
        'low_quality': 0,
        'q_type_counts': defaultdict(int)
    }
    
    quality_issues = []
    
    for idx, row in tqdm(meta.iterrows(), total=len(meta), desc="참가자 처리"):
        pid = str(row['Participant_ID'])
        label = int(row['Binary'])
        
        # 1. CSV에서 발화 추출
        utterances = preprocessor.process_transcript(pid)
        if not utterances:
            stats['failed'] += 1
            continue
        
        # 2. 오디오 로드 (전체 파일 한 번만)
        audio_path = os.path.join(BASE_PATH, f"{pid}_P", f"{pid}_AUDIO.wav")
        if not os.path.exists(audio_path):
            stats['failed'] += 1
            continue
        
        try:
            y_full, _ = librosa.load(audio_path, sr=SR)
            # 노이즈 제거
            y_full = nr.reduce_noise(y=y_full, sr=SR, stationary=True, prop_decrease=0.8)
        except Exception as e:
            stats['failed'] += 1
            continue
        
        # 3. 각 발화별 특징 추출
        processed_utterances = []
        
        for utt in utterances:
            # 오디오 추출
            start_sample = int(utt['start'] * SR)
            end_sample = int(utt['end'] * SR)
            
            if start_sample >= end_sample or end_sample > len(y_full):
                continue
            
            audio_segment = y_full[start_sample:end_sample]
            
            # 품질 검사
            issues = check_audio_quality(audio_segment, SR)
            if issues:
                quality_issues.append({
                    'pid': pid,
                    'duration': utt['duration'],
                    'issues': issues
                })
                stats['low_quality'] += 1
                # 심각한 문제 아니면 계속 진행
                if len(issues) > 2:  # 2개 이상 문제면 스킵
                    continue
            
            # Wav2Vec 특징 추출
            wav2vec_feat = extract_wav2vec(audio_segment, SR)
            if wav2vec_feat is None:
                continue
            
            # TTR 계산
            ttr = get_ttr(utt['text'])
            
            # Q-type ID 변환
            q_type_id = Q_TYPE_MAPPING.get(utt['q_type'], Q_TYPE_MAPPING['other'])
            
            processed_utterances.append({
                'wav2vec': wav2vec_feat,  # [768]
                'q_type': utt['q_type'],
                'q_type_id': q_type_id,
                'ttr': ttr,
                'duration': utt['duration'],
                'text': utt['text']
            })
            
            stats['total_utterances'] += 1
            stats['q_type_counts'][utt['q_type']] += 1
        
        if not processed_utterances:
            stats['failed'] += 1
            continue
        
        # 4. 참가자 데이터 저장
        dataset[pid] = {
            'label': label,
            'utterances': processed_utterances,
            'num_utterances': len(processed_utterances)
        }
        
        stats['processed'] += 1
    
    # 통계 출력
    print(f"\n{'='*70}")
    print(f"✅ 전처리 완료!")
    print(f"{'='*70}")
    print(f"처리 성공: {stats['processed']}명")
    print(f"처리 실패: {stats['failed']}명")
    print(f"총 발화: {stats['total_utterances']}개")
    print(f"품질 이슈: {stats['low_quality']}개 발화")
    
    print(f"\nQuestion Type 분포:")
    for q_type, count in sorted(stats['q_type_counts'].items(), key=lambda x: -x[1]):
        percentage = (count / stats['total_utterances']) * 100
        print(f"  {q_type:15s}: {count:5d}개 ({percentage:5.1f}%)")
    
    # 품질 이슈 샘플 출력
    if quality_issues:
        print(f"\n품질 이슈 샘플 (상위 10개):")
        for issue in quality_issues[:10]:
            print(f"  PID {issue['pid']}: {issue['duration']:.1f}초 - {', '.join(issue['issues'])}")
    
    return dataset


# =============================================================================
# 실행 및 저장
# =============================================================================
if __name__ == "__main__":
    # 전처리 실행
    dataset = run_preprocessing_pipeline()
    
    # 저장
    output_path = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")
    with open(output_path, 'wb') as f:
        pickle.dump(dataset, f)
    
    print(f"\n💾 데이터 저장 완료: {output_path}")
    
    # 샘플 데이터 확인
    sample_pid = list(dataset.keys())[0]
    sample = dataset[sample_pid]
    
    print(f"\n{'='*70}")
    print(f"📋 샘플 데이터 구조 (PID: {sample_pid})")
    print(f"{'='*70}")
    print(f"Label: {sample['label']}")
    print(f"Num Utterances: {sample['num_utterances']}")
    print(f"\n첫 번째 발화:")
    utt = sample['utterances'][0]
    print(f"  - Q-type: {utt['q_type']} (ID: {utt['q_type_id']})")
    print(f"  - Duration: {utt['duration']:.2f}초")
    print(f"  - TTR: {utt['ttr']:.3f}")
    print(f"  - Wav2Vec Shape: {utt['wav2vec'].shape}")
    print(f"  - Text: {utt['text'][:50]}...")

⏳ Wav2Vec 2.0 모델 로딩 중... (Device: cuda)


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ 모델 로드 완료!

🚀 전처리 시작: 186명



참가자 처리: 100%|██████████| 186/186 [27:56<00:00,  9.02s/it]



✅ 전처리 완료!
처리 성공: 186명
처리 실패: 0명
총 발화: 28580개
품질 이슈: 18248개 발화

Question Type 분포:
  background     : 11743개 ( 41.1%)
  casual         :  8313개 ( 29.1%)
  emotional      :  5607개 ( 19.6%)
  clinical       :  2917개 ( 10.2%)

품질 이슈 샘플 (상위 10개):
  PID 300: 3.1초 - 무음 비율 높음: 83.51%
  PID 300: 0.8초 - 무음 비율 높음: 100.00%
  PID 300: 1.5초 - 무음 비율 높음: 100.00%
  PID 300: 3.3초 - 무음 비율 높음: 100.00%
  PID 300: 0.6초 - 무음 비율 높음: 100.00%, 볼륨 너무 작음: 0.0087
  PID 300: 0.7초 - 무음 비율 높음: 100.00%
  PID 300: 0.8초 - 무음 비율 높음: 100.00%
  PID 300: 1.1초 - 무음 비율 높음: 94.12%
  PID 300: 1.5초 - 무음 비율 높음: 95.83%
  PID 300: 0.9초 - 무음 비율 높음: 100.00%

💾 데이터 저장 완료: D:\depression_dataset(DAIC-WOZ)\preprocessed_utterance_dataset.pkl

📋 샘플 데이터 구조 (PID: 300)
Label: 0
Num Utterances: 83

첫 번째 발화:
  - Q-type: casual (ID: 0)
  - Duration: 0.85초
  - TTR: 1.000
  - Wav2Vec Shape: (768,)
  - Text: good...


**모델 input 테스트

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# =============================================================================
# 모델 하이퍼파라미터
# =============================================================================
WAV2VEC_DIM = 768
Q_TYPE_VOCAB_SIZE = 5  # casual, background, emotional, clinical, other
Q_TYPE_EMBED_DIM = 32
TTR_DIM = 1

# Transformer 설정
D_MODEL = 256              # Transformer hidden dimension
NHEAD = 8                  # Attention heads
NUM_ENCODER_LAYERS = 3     # Transformer encoder layers
DIM_FEEDFORWARD = 512      # FFN dimension
DROPOUT = 0.3

# Input dimension = Wav2Vec(768) + Q-type(32) + TTR(1) = 801
INPUT_DIM = WAV2VEC_DIM + Q_TYPE_EMBED_DIM + TTR_DIM


# =============================================================================
# Positional Encoding
# =============================================================================
class PositionalEncoding(nn.Module):
    """위치 정보 인코딩"""
    
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # 위치 인코딩 계산
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, d_model]
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Transformer 기반 우울증 감지 모델
# =============================================================================
class TransformerDepressionModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(TransformerDepressionModel, self).__init__()
        
        self.d_model = d_model
        
        # =====================================================================
        # Input Embeddings
        # =====================================================================
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # Input projection: 801 → 256
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # =====================================================================
        # Transformer Encoder
        # =====================================================================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # =====================================================================
        # [CLS] token (학습 가능한 토큰)
        # =====================================================================
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # =====================================================================
        # Classifier
        # =====================================================================
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttrs, batch_q_type_ids, num_utterances_list):
        """
        Args:
            batch_wav2vec: [total_utterances, 768]
            batch_ttrs: [total_utterances]
            batch_q_type_ids: [total_utterances]
            num_utterances_list: [batch_size] - 각 참가자의 발화 수
        
        Returns:
            logits: [batch_size, 1]
            attention_weights: [batch_size, num_utterances] (해석용)
        """
        
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        # =====================================================================
        # 1. Feature 결합
        # =====================================================================
        # Q-type embedding
        q_type_embs = self.q_type_embedding(batch_q_type_ids)  # [total_utterances, 32]
        
        # TTR 확장
        ttrs_expanded = batch_ttrs.unsqueeze(1)  # [total_utterances, 1]
        
        # 결합: [768] + [32] + [1] = [801]
        combined_features = torch.cat([
            batch_wav2vec,
            q_type_embs,
            ttrs_expanded
        ], dim=1)  # [total_utterances, 801]
        
        # =====================================================================
        # 2. 참가자별로 재구성
        # =====================================================================
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # =====================================================================
        # 3. 패딩 및 마스크 생성
        # =====================================================================
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            # 패딩
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            # Attention mask (True = 무시할 위치)
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        # Stack
        padded_sequences = torch.stack(padded_sequences)  # [batch_size, max_num_utterances, 801]
        attention_masks = torch.stack(attention_masks)     # [batch_size, max_num_utterances]
        
        # =====================================================================
        # 4. Input Projection
        # =====================================================================
        x = self.input_projection(padded_sequences)  # [batch_size, max_num_utterances, 256]
        
        # =====================================================================
        # 5. [CLS] token 추가
        # =====================================================================
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # [batch_size, 1, 256]
        x = torch.cat([cls_tokens, x], dim=1)  # [batch_size, max_num_utterances+1, 256]
        
        # Mask도 확장 (CLS는 masking 안 함)
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)  # [batch_size, max_num_utterances+1]
        
        # =====================================================================
        # 6. Positional Encoding
        # =====================================================================
        x = self.pos_encoder(x)
        
        # =====================================================================
        # 7. Transformer Encoder
        # =====================================================================
        # src_key_padding_mask: True인 위치는 무시됨
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )  # [batch_size, max_num_utterances+1, 256]
        
        # =====================================================================
        # 8. [CLS] token 추출 (첫 번째 토큰)
        # =====================================================================
        cls_output = encoded[:, 0, :]  # [batch_size, 256]
        
        # =====================================================================
        # 9. Classification
        # =====================================================================
        logits = self.classifier(cls_output)  # [batch_size, 1]
        
        # =====================================================================
        # 10. Attention weights 추출 (해석용)
        # =====================================================================
        # 마지막 레이어의 self-attention weights 추출
        # Note: 실제로는 encoder layer의 attention을 저장해야 하지만,
        # 여기서는 간단히 utterance features의 norm을 사용
        utterance_features = encoded[:, 1:, :]  # [batch_size, max_num_utterances, 256]
        attention_weights = torch.norm(utterance_features, dim=2)  # [batch_size, max_num_utterances]
        
        # Masking된 부분은 0으로
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        
        # Softmax (해석용)
        # 실제 attention이 아니라 각 발화의 중요도 근사값
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# 모델 초기화
# =============================================================================
def initialize_transformer_model(device='cuda'):
    """Transformer 모델 초기화"""
    model = TransformerDepressionModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    )
    
    model = model.to(device)
    
    # 파라미터 수 계산
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"{'='*70}")
    print(f"🤖 Transformer 모델 초기화 완료")
    print(f"{'='*70}")
    print(f"총 파라미터 수: {total_params:,}")
    print(f"학습 가능 파라미터 수: {trainable_params:,}")
    print(f"Device: {device}")
    print(f"{'='*70}")
    
    return model


# =============================================================================
# 테스트 코드
# =============================================================================
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = initialize_transformer_model(device)
    
    # 더미 데이터
    batch_size = 4
    num_utterances_list = [15, 20, 10, 18]
    total_utterances = sum(num_utterances_list)
    
    batch_wav2vec = torch.randn(total_utterances, 768).to(device)
    batch_ttrs = torch.rand(total_utterances).to(device)
    batch_q_type_ids = torch.randint(0, 5, (total_utterances,)).to(device)
    
    print(f"\n{'='*70}")
    print(f"🧪 Forward Pass 테스트")
    print(f"{'='*70}")
    
    with torch.no_grad():
        logits, attention_weights = model(
            batch_wav2vec,
            batch_ttrs,
            batch_q_type_ids,
            num_utterances_list
        )
    
    print(f"Input:")
    print(f"  - Batch Size: {batch_size}")
    print(f"  - Total Utterances: {total_utterances}")
    print(f"  - Num Utterances per Participant: {num_utterances_list}")
    print(f"\nOutput:")
    print(f"  - Logits Shape: {logits.shape}")
    print(f"  - Attention Weights Shape: {attention_weights.shape}")
    print(f"  - Sample Logits: {logits[:3].squeeze().tolist()}")
    print(f"\n✅ 모델 테스트 통과!")

🤖 Transformer 모델 초기화 완료
총 파라미터 수: 1,820,577
학습 가능 파라미터 수: 1,820,577
Device: cuda

🧪 Forward Pass 테스트
Input:
  - Batch Size: 4
  - Total Utterances: 63
  - Num Utterances per Participant: [15, 20, 10, 18]

Output:
  - Logits Shape: torch.Size([4, 1])
  - Attention Weights Shape: torch.Size([4, 20])
  - Sample Logits: [-0.22017383575439453, 0.014878042042255402, -0.23515728116035461]

✅ 모델 테스트 통과!


**모델 학습, 검증**

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import os
import math
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 설정
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
PREPROCESSED_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")

# 모델 하이퍼파라미터
WAV2VEC_DIM = 768
Q_TYPE_VOCAB_SIZE = 5
Q_TYPE_EMBED_DIM = 32
TTR_DIM = 1
INPUT_DIM = WAV2VEC_DIM + Q_TYPE_EMBED_DIM + TTR_DIM  # 801

D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.3

# 학습 하이퍼파라미터
BATCH_SIZE = 8
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# ⭐ 불균형 처리 전략 개선 (Version 3 - 정상 구별 강화)
USE_FOCAL_LOSS = True  # Focal Loss 사용 (필수)
FOCAL_ALPHA = 0.45     # 0.68 → 0.45 (정상에 더 집중)
FOCAL_GAMMA = 1.5      # 2.5 → 1.5 (덜 aggressive)
LABEL_SMOOTHING = 0.08 # 0.12 → 0.08 (과신 방지)

# ⭐ Threshold 동적 조정 옵션
DYNAMIC_THRESHOLD = True    # Validation에서 최적 threshold 찾기
MIN_RECALL_THRESHOLD = 0.70 # 0.75 → 0.70 (최소 우울증 탐지율)

EARLY_STOPPING_PATIENCE = 15  # 12 → 15 (더 기다리기)
GRADIENT_CLIP = 1.0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# =============================================================================
# 개선된 Loss Functions
# =============================================================================
class FocalLoss(nn.Module):
    """
    Focal Loss: 어려운 샘플에 집중, 극단적 예측 방지
    
    FL(pt) = -alpha * (1-pt)^gamma * log(pt)
    
    - alpha: 클래스 균형 (우울증에 더 집중)
    - gamma: 어려운 샘플 집중도 (높을수록 쉬운 샘플 무시)
    """
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        # Label smoothing: 0 → 0.05, 1 → 0.95
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        # BCE loss
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        
        # pt: 정답일 확률
        pt = torch.exp(-bce_loss)
        
        # Focal weight: (1-pt)^gamma
        # 확신있게 맞춘 샘플(pt ≈ 1)의 loss 감소
        focal_weight = (1 - pt) ** self.gamma
        
        # Alpha balancing
        alpha_weight = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        
        # Final loss
        focal_loss = alpha_weight * focal_weight * bce_loss
        
        return focal_loss.mean()


class BalancedBCELoss(nn.Module):
    """
    개선된 BCE Loss with:
    - Label smoothing: 과신 방지
    - Pos weight: 불균형 처리
    - Penalty for extreme predictions: 전부 우울증 찍기 방지
    """
    def __init__(self, pos_weight=2.32, label_smoothing=0.1):
        super(BalancedBCELoss, self).__init__()
        self.pos_weight = pos_weight
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        # Label smoothing
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + self.label_smoothing / 2
        
        # BCE with pos_weight
        loss = F.binary_cross_entropy_with_logits(
            logits, 
            targets,
            pos_weight=torch.tensor([self.pos_weight], device=logits.device),
            reduction='none'
        )
        
        # Diversity penalty: 예측 분산이 낮으면 패널티
        # (전부 0 또는 전부 1로 예측하는 것 방지)
        probs = torch.sigmoid(logits)
        diversity_penalty = -torch.var(probs) * 0.1  # 분산이 낮으면 패널티
        
        return loss.mean() + diversity_penalty


class PositionalEncoding(nn.Module):
    """위치 정보 인코딩"""
    
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =============================================================================
# Transformer 기반 우울증 감지 모델
# =============================================================================
class TransformerDepressionModel(nn.Module):
    def __init__(
        self,
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ):
        super(TransformerDepressionModel, self).__init__()
        
        self.d_model = d_model
        
        # Question Type 임베딩
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )
        
        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttrs, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        # Feature 결합
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        combined_features = torch.cat([
            batch_wav2vec,
            q_type_embs,
            ttrs_expanded
        ], dim=1)
        
        # 참가자별로 재구성
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        # 패딩 및 마스크
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(
                    max_num_utterances - num_utts,
                    seq.size(1),
                    device=device
                )
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        # Input Projection
        x = self.input_projection(padded_sequences)
        
        # [CLS] token 추가
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        # Positional Encoding
        x = self.pos_encoder(x)
        
        # Transformer Encoder
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=extended_mask
        )
        
        # [CLS] token 추출
        cls_output = encoded[:, 0, :]
        
        # Classification
        logits = self.classifier(cls_output)
        
        # Attention weights (해석용)
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# PyTorch Dataset
# =============================================================================
class UtteranceDataset(Dataset):
    def __init__(self, participant_data):
        """
        Args:
            participant_data: dict of {pid: {'label': ..., 'utterances': [...]}}
        """
        self.data = []
        
        for pid, info in participant_data.items():
            self.data.append({
                'pid': pid,
                'label': info['label'],
                'utterances': info['utterances'],
                'num_utterances': info['num_utterances']
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


# =============================================================================
# Collate Function
# =============================================================================
def collate_fn(batch):
    """
    배치 내 참가자들의 발화 수가 다르므로 동적 처리
    
    Returns:
        batch_wav2vec: [total_utterances, 768]
        batch_ttrs: [total_utterances]
        batch_q_type_ids: [total_utterances]
        batch_labels: [batch_size, 1]
        num_utterances_list: [batch_size]
    """
    
    batch_labels = []
    all_utterances = []
    num_utterances_list = []
    
    for item in batch:
        batch_labels.append(item['label'])
        all_utterances.extend(item['utterances'])
        num_utterances_list.append(item['num_utterances'])
    
    # 특징 추출
    batch_wav2vec = []
    batch_ttrs = []
    batch_q_type_ids = []
    
    for utt in all_utterances:
        batch_wav2vec.append(utt['wav2vec'])
        batch_ttrs.append(utt['ttr'])
        batch_q_type_ids.append(utt['q_type_id'])
    
    # Tensor 변환
    batch_wav2vec = torch.FloatTensor(np.array(batch_wav2vec))  # [total_utterances, 768]
    batch_ttrs = torch.FloatTensor(batch_ttrs)  # [total_utterances]
    batch_q_type_ids = torch.LongTensor(batch_q_type_ids)  # [total_utterances]
    batch_labels = torch.FloatTensor(batch_labels).unsqueeze(1)  # [batch_size, 1]
    
    return {
        'batch_wav2vec': batch_wav2vec,
        'batch_ttrs': batch_ttrs,
        'batch_q_type_ids': batch_q_type_ids,
        'batch_labels': batch_labels,
        'num_utterances_list': num_utterances_list
    }


# =============================================================================
# 데이터 로드 및 분할
# =============================================================================
def load_and_split_data():
    """전처리된 데이터 로드 및 Train/Val/Test 분할 (메타데이터 Group 기준)"""
    print(f"{'='*70}")
    print(f"📂 데이터 로드 중...")
    print(f"{'='*70}")
    
    with open(PREPROCESSED_DATA_PATH, 'rb') as f:
        dataset = pickle.load(f)
    
    # 메타데이터 로드하여 Group 정보 사용
    meta_df = pd.read_csv(os.path.join(BASE_PATH, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # Group 열을 기준으로 분할
    train_pids = meta_df[meta_df['Group'] == 'Train']['Participant_ID'].tolist()
    val_pids = meta_df[meta_df['Group'] == 'Validation']['Participant_ID'].tolist()
    test_pids = meta_df[meta_df['Group'] == 'Test']['Participant_ID'].tolist()
    
    # dataset에 있는 PID만 사용 (전처리된 데이터 기준)
    train_pids = [pid for pid in train_pids if pid in dataset]
    val_pids = [pid for pid in val_pids if pid in dataset]
    test_pids = [pid for pid in test_pids if pid in dataset]
    
    # 라벨 추출
    train_labels = [dataset[pid]['label'] for pid in train_pids]
    val_labels = [dataset[pid]['label'] for pid in val_pids]
    test_labels = [dataset[pid]['label'] for pid in test_pids]
    
    print(f"총 참가자 수: {len(dataset)}")
    print(f"  - 정상 (0): {sum(1 for d in dataset.values() if d['label'] == 0)}")
    print(f"  - 우울증 (1): {sum(1 for d in dataset.values() if d['label'] == 1)}")
    
    print(f"\n메타데이터 기준 분할:")
    print(f"  - Train: {len(train_pids)} (0: {train_labels.count(0)}, 1: {train_labels.count(1)})")
    print(f"  - Val:   {len(val_pids)} (0: {val_labels.count(0)}, 1: {val_labels.count(1)})")
    print(f"  - Test:  {len(test_pids)} (0: {test_labels.count(0)}, 1: {test_labels.count(1)})")
    
    # 데이터셋 생성
    train_data = {pid: dataset[pid] for pid in train_pids}
    val_data = {pid: dataset[pid] for pid in val_pids}
    test_data = {pid: dataset[pid] for pid in test_pids}
    
    train_dataset = UtteranceDataset(train_data)
    val_dataset = UtteranceDataset(val_data)
    test_dataset = UtteranceDataset(test_data)
    
    # DataLoader 생성
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader, test_loader


# =============================================================================
# 평가 함수
# =============================================================================
def evaluate(model, dataloader, criterion, threshold=0.5, return_all_thresholds=False):
    """
    모델 평가 - 동적 threshold 지원
    
    Args:
        return_all_thresholds: True면 여러 threshold에 대한 결과 반환
    """
    model.eval()
    
    all_labels = []
    all_probs = []
    all_preds = []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            
            all_labels.extend(batch_labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
    
    # 기본 메트릭
    avg_loss = total_loss / len(dataloader)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    
    # 동적 threshold 탐색
    if return_all_thresholds and DYNAMIC_THRESHOLD:
        best_threshold, best_metrics = find_optimal_threshold(
            all_labels, 
            all_probs,
            min_recall=MIN_RECALL_THRESHOLD
        )
        return {
            'loss': avg_loss,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'labels': all_labels,
            'probs': all_probs,
            'preds': all_preds,
            'best_threshold': best_threshold,
            'best_metrics': best_metrics
        }
    
    return {
        'loss': avg_loss,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'labels': all_labels,
        'probs': all_probs,
        'preds': all_preds
    }


def find_optimal_threshold(labels, probs, min_recall=0.75):
    """
    F1 최대화하는 threshold 찾기 (최소 recall 제약 포함)
    
    Args:
        min_recall: 최소 우울증 탐지율 (이 이상 유지해야 함)
    """
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    # 0.1 ~ 0.7 범위에서 탐색 (더 넓게)
    for thresh in np.arange(0.1, 0.7, 0.02):
        preds = (np.array(probs) > thresh).astype(int)
        
        # 메트릭 계산
        f1 = f1_score(labels, preds)
        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        
        # 최소 recall 제약 확인
        if recall < min_recall:
            continue
        
        # F1 개선되면 업데이트
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'threshold': thresh
            }
    
    return best_threshold, best_metrics


# =============================================================================
# 최적 Threshold 찾기
# =============================================================================
def find_best_threshold(labels, probs, min_thresh=0.1, max_thresh=0.9, step=0.05):
    """F1 score를 최대화하는 threshold 찾기"""
    best_f1 = 0.0
    best_threshold = 0.5
    
    thresholds = np.arange(min_thresh, max_thresh, step)
    f1_scores = []
    
    for thresh in thresholds:
        preds = (np.array(probs) > thresh).astype(int)
        f1 = f1_score(labels, preds)
        f1_scores.append(f1)
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
    
    # 시각화
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, f1_scores, marker='o')
    plt.axvline(best_threshold, color='r', linestyle='--', label=f'Best: {best_threshold:.2f}')
    plt.xlabel('Threshold')
    plt.ylabel('F1 Score')
    plt.title('F1 Score vs Threshold')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(BASE_PATH, 'threshold_tuning_transformer.png'))
    plt.close()
    
    print(f"\n{'='*70}")
    print(f"🎯 최적 Threshold 탐색 결과")
    print(f"{'='*70}")
    print(f"Best Threshold: {best_threshold:.3f}")
    print(f"Best F1 Score: {best_f1:.4f}")
    
    return best_threshold, best_f1


# =============================================================================
# Training Loop
# =============================================================================
def train_model(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    """모델 학습 - 극단적 예측 방지 버전"""
    
    # =========================================================================
    # Loss function 선택
    # =========================================================================
    if USE_FOCAL_LOSS:
        criterion = FocalLoss(
            alpha=FOCAL_ALPHA,
            gamma=FOCAL_GAMMA,
            label_smoothing=LABEL_SMOOTHING
        )
        print(f"📍 Using Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA})")
    else:
        criterion = BalancedBCELoss(
            pos_weight=2.32,
            label_smoothing=LABEL_SMOOTHING
        )
        print(f"📍 Using Balanced BCE Loss (pos_weight=2.32)")
    
    # =========================================================================
    # Optimizer & Scheduler
    # =========================================================================
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    
    # Warmup + Cosine Annealing
    warmup_epochs = 5
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.1,
        end_factor=1.0,
        total_iters=warmup_epochs
    )
    
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs - warmup_epochs,
        eta_min=1e-6
    )
    
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs]
    )
    
    # =========================================================================
    # Early stopping - F1 기준 (Precision/Recall 균형 고려)
    # =========================================================================
    best_val_f1 = 0.0
    best_val_precision = 0.0
    best_val_recall = 0.0
    patience_counter = 0
    
    # 기록
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1': [],
        'val_precision': [],
        'val_recall': [],
        'val_specificity': []  # 정상인을 정상으로 맞추는 비율
    }
    
    print(f"\n{'='*70}")
    print(f"🚀 학습 시작")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        # =====================================================================
        # Training
        # =====================================================================
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            batch_wav2vec = batch['batch_wav2vec'].to(DEVICE)
            batch_ttrs = batch['batch_ttrs'].to(DEVICE)
            batch_q_type_ids = batch['batch_q_type_ids'].to(DEVICE)
            batch_labels = batch['batch_labels'].to(DEVICE)
            
            optimizer.zero_grad()
            logits, _ = model(
                batch_wav2vec,
                batch_ttrs,
                batch_q_type_ids,
                batch['num_utterances_list']
            )
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        avg_train_loss = train_loss / len(train_loader)
        
        # =====================================================================
        # Validation - 동적 threshold로 평가
        # =====================================================================
        val_results = evaluate(
            model, 
            val_loader, 
            criterion, 
            threshold=0.5,
            return_all_thresholds=True
        )
        
        # 최적 threshold 사용
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            best_thresh = val_results['best_threshold']
            best_met = val_results['best_metrics']
            
            # 최적 threshold로 재계산
            optimal_preds = (np.array(val_results['probs']) > best_thresh).astype(int)
            optimal_f1 = f1_score(val_results['labels'], optimal_preds)
            optimal_prec = precision_score(val_results['labels'], optimal_preds, zero_division=0)
            optimal_rec = recall_score(val_results['labels'], optimal_preds, zero_division=0)
            
            # 기록
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(optimal_f1)
            history['val_precision'].append(optimal_prec)
            history['val_recall'].append(optimal_rec)
        else:
            # 고정 threshold (0.5)
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_results['loss'])
            history['val_f1'].append(val_results['f1'])
            history['val_precision'].append(val_results['precision'])
            history['val_recall'].append(val_results['recall'])
            best_thresh = 0.5
            optimal_f1 = val_results['f1']
            optimal_prec = val_results['precision']
            optimal_rec = val_results['recall']
        
        # Specificity 계산
        optimal_preds_list = (np.array(val_results['probs']) > best_thresh).astype(int)
        tn = sum((l == 0 and p == 0) for l, p in zip(val_results['labels'], optimal_preds_list))
        fp = sum((l == 0 and p == 1) for l, p in zip(val_results['labels'], optimal_preds_list))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        history['val_specificity'].append(specificity)
        
        scheduler.step()
        
        # =====================================================================
        # 출력 - 최적 threshold 정보 포함
        # =====================================================================
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss:  {avg_train_loss:.4f}")
        print(f"  Val Loss:    {val_results['loss']:.4f}")
        
        if DYNAMIC_THRESHOLD and 'best_threshold' in val_results:
            print(f"  Optimal Threshold: {best_thresh:.3f}")
        
        print(f"  Val F1:      {optimal_f1:.4f}")
        print(f"  Val Prec:    {optimal_prec:.4f}")
        print(f"  Val Recall:  {optimal_rec:.4f}")
        print(f"  Val Spec:    {specificity:.4f}  ← 정상 구별")
        print(f"  LR:          {optimizer.param_groups[0]['lr']:.6f}")
        
        # =====================================================================
        # 극단적 예측 경고 (기준 완화)
        # =====================================================================
        if optimal_rec > 0.98 and optimal_prec < 0.35:
            print(f"  ⚠️  경고: 여전히 우울증 편향 (Rec={optimal_rec:.2f}, Prec={optimal_prec:.2f})")
            print(f"      → FOCAL_ALPHA 감소 권장 (현재 {FOCAL_ALPHA})")
        
        if specificity < 0.2:
            print(f"  ⚠️  경고: 정상 구별 실패 (Spec={specificity:.2f})")
        
        # =====================================================================
        # Early Stopping - 조건 대폭 완화
        # =====================================================================
        is_balanced = (
            optimal_prec > 0.25 and      # 0.35 → 0.25 (크게 완화)
            optimal_rec > 0.60 and       # 0.65 → 0.60 (완화)
            specificity > 0.25           # 0.40 → 0.25 (크게 완화!)
        )
        
        if optimal_f1 > best_val_f1 and is_balanced:
            best_val_f1 = optimal_f1
            best_val_precision = optimal_prec
            best_val_recall = optimal_rec
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': best_val_f1,
                'val_precision': best_val_precision,
                'val_recall': best_val_recall,
                'val_specificity': specificity,
                'optimal_threshold': best_thresh,
                'history': history
            }, os.path.join(BASE_PATH, 'best_transformer_model.pt'))
            
            print(f"  ✅ Best model saved! (F1: {best_val_f1:.4f}, "
                  f"Prec: {best_val_precision:.4f}, Rec: {best_val_recall:.4f}, "
                  f"Thresh: {best_thresh:.3f})")
        else:
            patience_counter += 1
            if not is_balanced:
                print(f"  ⚠️  Balanced 조건 미충족:")
                print(f"      Prec={optimal_prec:.2f} (>0.25?), Rec={optimal_rec:.2f} (>0.60?), Spec={specificity:.2f} (>0.25?)")
            print(f"  ⏳ No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
            
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"\n⚠️  Early stopping triggered!")
                break
        
        print("-" * 70)
    
    return history


# =============================================================================
# 시각화
# =============================================================================
def plot_training_history(history):
    """학습 곡선 시각화"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss')
    axes[0, 0].plot(history['val_loss'], label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training & Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # F1 Score
    axes[0, 1].plot(history['val_f1'], label='Val F1', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('Validation F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Precision
    axes[1, 0].plot(history['val_precision'], label='Val Precision', color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].set_title('Validation Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Recall
    axes[1, 1].plot(history['val_recall'], label='Val Recall', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Validation Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, 'training_history_transformer.png'))
    plt.close()


def plot_confusion_matrix(labels, preds, title='Confusion Matrix'):
    """Confusion Matrix 시각화"""
    cm = confusion_matrix(labels, preds)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.savefig(os.path.join(BASE_PATH, f'{title.lower().replace(" ", "_")}_transformer.png'))
    plt.close()


# =============================================================================
# 메인 실행
# =============================================================================
if __name__ == "__main__":
    # 데이터 로드
    train_loader, val_loader, test_loader = load_and_split_data()
    
    # 모델 초기화
    model = TransformerDepressionModel(
        input_dim=INPUT_DIM,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_encoder_layers=NUM_ENCODER_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        dropout=DROPOUT,
        q_type_vocab_size=Q_TYPE_VOCAB_SIZE,
        q_type_embed_dim=Q_TYPE_EMBED_DIM
    ).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*70}")
    print(f"🤖 모델 초기화 완료")
    print(f"{'='*70}")
    print(f"총 파라미터 수: {total_params:,}")
    print(f"Device: {DEVICE}")
    print(f"{'='*70}\n")
    
    # 학습
    history = train_model(model, train_loader, val_loader)
    
    # 학습 곡선 시각화
    plot_training_history(history)
    print(f"\n✅ 학습 곡선 저장")
    
    # 최적 Threshold 찾기 (개선된 버전)
    checkpoint = torch.load(os.path.join(BASE_PATH, 'best_transformer_model.pt'))
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # 저장된 최적 threshold 확인
    if 'optimal_threshold' in checkpoint:
        saved_threshold = checkpoint['optimal_threshold']
        print(f"💡 모델에 저장된 최적 threshold: {saved_threshold:.3f}")
    else:
        saved_threshold = 0.5
    
    # Validation으로 최종 확인
    val_results = evaluate(model, val_loader, nn.BCEWithLogitsLoss(), 
                          threshold=saved_threshold, return_all_thresholds=True)
    
    if 'best_threshold' in val_results:
        best_threshold = val_results['best_threshold']
        best_met = val_results['best_metrics']
        print(f"💡 최종 검증 threshold: {best_threshold:.3f}")
        print(f"   → F1: {best_met['f1']:.4f}, Prec: {best_met['precision']:.4f}, Rec: {best_met['recall']:.4f}")
    else:
        best_threshold = saved_threshold
    
    # 최종 Test 평가
    print(f"\n{'='*70}")
    print(f"📊 최종 Test Set 평가 (Threshold: {best_threshold:.3f})")
    print(f"{'='*70}\n")
    
    test_results = evaluate(model, test_loader, nn.BCEWithLogitsLoss(), threshold=best_threshold)
    
    print(f"Test Loss:      {test_results['loss']:.4f}")
    print(f"Test F1:        {test_results['f1']:.4f}")
    print(f"Test Precision: {test_results['precision']:.4f}")
    print(f"Test Recall:    {test_results['recall']:.4f}")
    
    # Confusion Matrix
    plot_confusion_matrix(test_results['labels'], test_results['preds'], 'Test Confusion Matrix')
    
    # Classification Report
    print(f"\n{'='*70}")
    print("분류 보고서:")
    print(f"{'='*70}")
    print(classification_report(test_results['labels'], test_results['preds'],
                                target_names=['Normal', 'Depression']))
    
    print(f"\n✅ 모든 학습 및 평가 완료!")

📂 데이터 로드 중...
총 참가자 수: 186
  - 정상 (0): 129
  - 우울증 (1): 57

메타데이터 기준 분할:
  - Train: 107 (0: 76, 1: 31)
  - Val:   33 (0: 21, 1: 12)
  - Test:  46 (0: 32, 1: 14)

🤖 모델 초기화 완료
총 파라미터 수: 1,820,577
Device: cuda

📍 Using Focal Loss (alpha=0.45, gamma=1.5)

🚀 학습 시작



Epoch 1/50: 100%|██████████| 14/14 [00:00<00:00, 22.42it/s, loss=0.2]  



Epoch 1/50
  Train Loss:  0.1655
  Val Loss:    0.1288
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000028
  ⚠️  경고: 정상 구별 실패 (Spec=0.00)
  ⚠️  Balanced 조건 미충족:
      Prec=0.36 (>0.25?), Rec=1.00 (>0.60?), Spec=0.00 (>0.25?)
  ⏳ No improvement (1/15)
----------------------------------------------------------------------


Epoch 2/50: 100%|██████████| 14/14 [00:00<00:00, 37.15it/s, loss=0.125]



Epoch 2/50
  Train Loss:  0.1250
  Val Loss:    0.1098
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000046
  ⚠️  경고: 정상 구별 실패 (Spec=0.00)
  ⚠️  Balanced 조건 미충족:
      Prec=0.36 (>0.25?), Rec=1.00 (>0.60?), Spec=0.00 (>0.25?)
  ⏳ No improvement (2/15)
----------------------------------------------------------------------


...


Epoch 13/50
  Train Loss:  0.1069
  Val Loss:    0.1119
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000092
  ⚠️  경고: 정상 구별 실패 (Spec=0.00)
  ⚠️  Balanced 조건 미충족:
      Prec=0.36 (>0.25?), Rec=1.00 (>0.60?), Spec=0.00 (>0.25?)
  ⏳ No improvement (13/15)
----------------------------------------------------------------------


Epoch 14/50: 100%|██████████| 14/14 [00:00<00:00, 37.38it/s, loss=0.0887]



Epoch 14/50
  Train Loss:  0.1063
  Val Loss:    0.1101
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000091
  ⚠️  경고: 정상 구별 실패 (Spec=0.00)
  ⚠️  Balanced 조건 미충족:
      Prec=0.36 (>0.25?), Rec=1.00 (>0.60?), Spec=0.00 (>0.25?)
  ⏳ No improvement (14/15)
----------------------------------------------------------------------


Epoch 15/50: 100%|██████████| 14/14 [00:00<00:00, 39.77it/s, loss=0.0936]



Epoch 15/50
  Train Loss:  0.1132
  Val Loss:    0.1113
  Optimal Threshold: 0.100
  Val F1:      0.5333
  Val Prec:    0.3636
  Val Recall:  1.0000
  Val Spec:    0.0000  ← 정상 구별
  LR:          0.000088
  ⚠️  경고: 정상 구별 실패 (Spec=0.00)
  ⚠️  Balanced 조건 미충족:
      Prec=0.36 (>0.25?), Rec=1.00 (>0.60?), Spec=0.00 (>0.25?)
  ⏳ No improvement (15/15)

⚠️  Early stopping triggered!

✅ 학습 곡선 저장
💡 모델에 저장된 최적 threshold: 0.340
💡 최종 검증 threshold: 0.340
   → F1: 0.7857, Prec: 0.6875, Rec: 0.9167

📊 최종 Test Set 평가 (Threshold: 0.340)

Test Loss:      0.5655
Test F1:        0.4865
Test Precision: 0.3913
Test Recall:    0.6429

분류 보고서:
              precision    recall  f1-score   support

      Normal       0.78      0.56      0.65        32
  Depression       0.39      0.64      0.49        14

    accuracy                           0.59        46
   macro avg       0.59      0.60      0.57        46
weighted avg       0.66      0.59      0.60        46


✅ 모든 학습 및 평가 완료!
